# Manual Input Verification

Verifies the manual input design from [`docs/architecture/manual_input.md`](../docs/architecture/manual_input.md)
without any aircraft physics.  Running the final cell opens a Qt window that shows the four
`AircraftCommand` fields updating in real time from keyboard or joystick.

This notebook implements Python analogs of the C++ classes described in the design document:

| Python class / function | C++ counterpart |
| --- | --- |
| `AxisMapping`, `JoystickConfig`, `KeyboardConfig` | Matching C++ structs in `JoystickInput.hpp` / `KeyboardInput.hpp` |
| `normalize_axis`, `apply_dead_zone`, `map_axis` | `JoystickInput::applyDeadZoneAndScale()` |
| `KeyboardState` | `KeyboardInput` (integrating command model) |
| `JoystickReader` | `JoystickInput` (SDL2 / pygame joystick wrapper) |
| `InputMonitorWindow` | No C++ equivalent — verification display only |

## Key bindings (keyboard mode)

| Key | Action |
| --- | --- |
| `↑` / `↓` | Increase / decrease normal load factor (n_z) |
| `←` / `→` | Roll left / right (rollRate) |
| `Q` / `E` | Yaw left / right (n_y) |
| `W` / `S` | Throttle up / down |
| `Space` | Center all axes to neutral |

Joystick is detected automatically.  If present, joystick takes priority.  On disconnect,
the display reverts to keyboard (neutral command until a key is pressed).

> **No C++ implementation required.**  This notebook runs entirely in Python and serves as
> a functional prototype and acceptance check for the design.

---

## Imports

In [ ]:
import math
import sys
import time
from dataclasses import dataclass, field

import matplotlib.figure as mpl_figure
import pygame
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from PySide6.QtCore import Qt, QTimer
from PySide6.QtGui import QKeyEvent
from PySide6.QtWidgets import QApplication, QLabel, QMainWindow, QVBoxLayout, QWidget

---
## Configuration

Python analogs of `AxisMapping`, `JoystickConfig`, and `KeyboardConfig` from the C++ design.
Edit these cells to match your device layout before running the display.

In [ ]:
@dataclass
class AxisMapping:
    """Mirrors C++ AxisMapping in JoystickInput.hpp."""
    sdl_axis_index: int   = 0      # pygame axis index (0-based)
    center_output:  float = 0.0    # command value at axis center
    scale:          float = 1.0    # command units per unit of normalized axis
    inverted:       bool  = False  # negate the normalized axis before scaling
    # pygame get_axis() already returns float in [-1, +1]; raw_min/raw_max
    # are expressed here as normalized floats instead of Sint16 values.
    raw_min: float = -1.0          # calibrated travel minimum (normalized)
    raw_max: float =  1.0          # calibrated travel maximum (normalized)


@dataclass
class JoystickConfig:
    """Mirrors C++ JoystickInputConfig.  Edit axis indices to match your device."""
    dead_zone_nd: float = 0.05

    # Axis mappings — defaults are illustrative; configure for your device.
    nz_axis:       AxisMapping = field(default_factory=lambda: AxisMapping(1,  1.0,  3.0, True))
    ny_axis:       AxisMapping = field(default_factory=lambda: AxisMapping(3,  0.0,  1.0, False))
    roll_axis:     AxisMapping = field(default_factory=lambda: AxisMapping(0,  0.0,  1.5708, False))
    throttle_axis: AxisMapping = field(default_factory=lambda: AxisMapping(2,  0.0,  1.0, True))

    min_nz_g:            float = -2.0
    max_nz_g:            float =  4.0
    max_ny_g:            float =  2.0
    max_roll_rate_rad_s: float =  1.5708   # π/2 rad/s
    center_button_index: int   =  0


@dataclass
class KeyboardConfig:
    """Mirrors C++ KeyboardInputConfig."""
    nz_rate_g_s:          float = 2.0
    ny_rate_g_s:          float = 1.0
    roll_rate_rate_rad_s2: float = 1.0
    throttle_rate_nd_s:   float = 0.5
    min_nz_g:             float = -2.0
    max_nz_g:             float =  4.0
    max_ny_g:             float =  2.0
    max_roll_rate_rad_s:  float =  1.5708
    idle_throttle_nd:     float =  0.05
    neutral_nz_g:         float =  1.0

---
## Axis Signal Processing

Python implementation of the normalization and dead-zone pipeline described in the design document.

$$r = \frac{\text{raw} - m}{\Delta/2}, \quad m = \frac{r_{\min}+r_{\max}}{2}, \quad \Delta = r_{\max}-r_{\min}$$

$$r' = \begin{cases} 0 & |r| < d \\ \operatorname{sign}(r)\,\dfrac{|r|-d}{1-d} & |r| \geq d \end{cases}$$

$$c = c_{\text{center}} + r' \times \text{scale}$$

In [ ]:
def normalize_axis(raw: float, mapping: AxisMapping) -> float:
    """Calibrate raw axis value and normalize to [-1, +1].

    Mirrors C++ JoystickInput::applyDeadZoneAndScale() step 1.
    pygame already normalizes to [-1, +1], but raw_min/raw_max allow
    trimming devices that only use part of that range (e.g. R/C transmitters).
    """
    mid = (mapping.raw_min + mapping.raw_max) / 2.0
    half_range = (mapping.raw_max - mapping.raw_min) / 2.0
    r = (raw - mid) / half_range
    r = max(-1.0, min(1.0, r))
    if mapping.inverted:
        r = -r
    return r


def apply_dead_zone(r: float, dead_zone: float) -> float:
    """Dead zone with continuity — mirrors C++ step 3.

    Output is 0 inside the dead zone, and linearly rescaled outside so that
    full deflection (|r| = 1) always maps to ±1 regardless of dead_zone size.
    """
    if abs(r) < dead_zone:
        return 0.0
    sign = 1.0 if r > 0.0 else -1.0
    return sign * (abs(r) - dead_zone) / (1.0 - dead_zone)


def map_axis(raw: float, mapping: AxisMapping, dead_zone: float) -> float:
    """Full pipeline: normalize → invert → dead zone → scale to command units."""
    r_norm = normalize_axis(raw, mapping)
    r_dz = apply_dead_zone(r_norm, dead_zone)
    return mapping.center_output + r_dz * mapping.scale

---
## Keyboard State — Integrating Model

Mirrors `KeyboardInput`: holding a key ramps the command at a configured rate.
Releasing the key leaves the command at its current value.  `Space` centers all axes.

In [ ]:
class KeyboardState:
    """Python analog of C++ KeyboardInput — integrating command model.

    Key state is received from Qt key press / release events in the
    display window.  update(dt_s) integrates the held keys each timer tick.
    """

    KEY_PITCH_UP      = Qt.Key.Key_Up
    KEY_PITCH_DOWN    = Qt.Key.Key_Down
    KEY_ROLL_RIGHT    = Qt.Key.Key_Right
    KEY_ROLL_LEFT     = Qt.Key.Key_Left
    KEY_YAW_RIGHT     = Qt.Key.Key_E
    KEY_YAW_LEFT      = Qt.Key.Key_Q
    KEY_THROTTLE_UP   = Qt.Key.Key_W
    KEY_THROTTLE_DOWN = Qt.Key.Key_S
    KEY_CENTER        = Qt.Key.Key_Space

    def __init__(self, cfg: KeyboardConfig):
        self.cfg = cfg
        self._held: set = set()
        self.reset()

    def reset(self) -> None:
        """Set all axes to the neutral / idle values."""
        c = self.cfg
        self.nz_g = c.neutral_nz_g
        self.ny_g = 0.0
        self.roll_rate_rad_s = 0.0
        self.throttle_nd = c.idle_throttle_nd

    def key_press(self, key: Qt.Key) -> None:
        self._held.add(key)

    def key_release(self, key: Qt.Key) -> None:
        self._held.discard(key)

    def update(self, dt_s: float) -> None:
        """Integrate held keys into command state.  Call once per timer tick."""
        if self.KEY_CENTER in self._held:
            self.reset()
            return

        c = self.cfg
        if self.KEY_PITCH_UP      in self._held: self.nz_g            += c.nz_rate_g_s * dt_s
        if self.KEY_PITCH_DOWN    in self._held: self.nz_g            -= c.nz_rate_g_s * dt_s
        if self.KEY_ROLL_RIGHT    in self._held: self.roll_rate_rad_s += c.roll_rate_rate_rad_s2 * dt_s
        if self.KEY_ROLL_LEFT     in self._held: self.roll_rate_rad_s -= c.roll_rate_rate_rad_s2 * dt_s
        if self.KEY_YAW_RIGHT     in self._held: self.ny_g            += c.ny_rate_g_s * dt_s
        if self.KEY_YAW_LEFT      in self._held: self.ny_g            -= c.ny_rate_g_s * dt_s
        if self.KEY_THROTTLE_UP   in self._held: self.throttle_nd     += c.throttle_rate_nd_s * dt_s
        if self.KEY_THROTTLE_DOWN in self._held: self.throttle_nd     -= c.throttle_rate_nd_s * dt_s

        self.nz_g            = max(c.min_nz_g,            min(c.max_nz_g,            self.nz_g))
        self.ny_g            = max(-c.max_ny_g,           min(c.max_ny_g,            self.ny_g))
        self.roll_rate_rad_s = max(-c.max_roll_rate_rad_s, min(c.max_roll_rate_rad_s, self.roll_rate_rad_s))
        self.throttle_nd     = max(0.0,                   min(1.0,                   self.throttle_nd))

---
## Joystick Reader

Wraps a `pygame.joystick.Joystick` — SDL2-based, matching the C++ `JoystickInput`.
Supports USB joysticks, game controllers, and R/C transmitters in USB HID mode.

In [ ]:
class JoystickReader:
    """Python analog of C++ JoystickInput.

    pygame.get_axis() already returns a float in [-1, +1], so the Sint16
    raw value step from the C++ design is replaced by float raw_min/raw_max
    calibration in AxisMapping.
    """

    def __init__(self, device_index: int = 0, cfg: JoystickConfig | None = None):
        self.cfg = cfg or JoystickConfig()
        self._joy = pygame.joystick.Joystick(device_index)
        self._joy.init()
        self.connected = True

    @property
    def name(self) -> str:
        return self._joy.get_name()

    @property
    def axis_count(self) -> int:
        return self._joy.get_numaxes()

    def raw_axes(self) -> list[float]:
        """Return all raw (pre-mapping) normalized axis values for diagnostics."""
        pygame.event.pump()
        return [self._joy.get_axis(i) for i in range(self._joy.get_numaxes())]

    def read(self) -> tuple[float, float, float, float]:
        """Return (nz_g, ny_g, roll_rate_rad_s, throttle_nd) with mapping applied."""
        pygame.event.pump()
        c = self.cfg

        def read_one(mapping: AxisMapping) -> float:
            raw = self._joy.get_axis(mapping.sdl_axis_index)
            return map_axis(raw, mapping, c.dead_zone_nd)

        nz       = max(c.min_nz_g,            min(c.max_nz_g,            read_one(c.nz_axis)))
        ny       = max(-c.max_ny_g,           min(c.max_ny_g,            read_one(c.ny_axis)))
        roll     = max(-c.max_roll_rate_rad_s, min(c.max_roll_rate_rad_s, read_one(c.roll_axis)))
        throttle = max(0.0,                   min(1.0,                   read_one(c.throttle_axis)))
        return nz, ny, roll, throttle

    def check_disconnect(self) -> None:
        """Process pending SDL events and update connected flag on JOYDEVICEREMOVED."""
        for event in pygame.event.get(pygame.JOYDEVICEREMOVED):
            if event.instance_id == self._joy.get_instance_id():
                self.connected = False

---
## Display — Gauge Drawing

In [ ]:
def draw_gauge(
    ax,
    value: float,
    v_min: float,
    v_max: float,
    v_neutral: float,
    label: str,
    unit: str,
    source_color: str,
) -> None:
    """Draw one horizontal gauge on a matplotlib Axes.

    The bar fills from the neutral position to the current value.  A vertical
    line marks neutral.  Color transitions from blue (near neutral) to orange
    (large deflection) to red (at the limit).
    """
    ax.clear()
    span = v_max - v_min

    # Background track
    ax.barh(0, span, left=v_min, height=0.55, color="#ebebeb", edgecolor="#c8c8c8", linewidth=0.5)

    # Choose bar color by normalized deflection from neutral
    deflection = abs(value - v_neutral) / (span / 2.0)
    if deflection < 0.25:
        bar_color = source_color
    elif deflection < 0.75:
        bar_color = "#FF9800"
    else:
        bar_color = "#F44336"

    # Value bar from neutral to current value
    ax.barh(0, value - v_neutral, left=v_neutral, height=0.55, color=bar_color, alpha=0.85)

    # Neutral marker
    ax.axvline(v_neutral, color="#303030", linewidth=1.5, zorder=4)

    # Value text (right-aligned inside the Axes)
    ax.text(
        0.99, 0.5,
        f"{value:+6.2f} {unit}",
        transform=ax.transAxes,
        va="center", ha="right",
        fontsize=11, fontfamily="monospace", fontweight="bold",
    )

    # Axis label (left-aligned)
    ax.set_title(label, fontsize=10, pad=3, loc="left", fontweight="bold", color="#404040")
    ax.set_xlim(v_min, v_max)
    ax.set_ylim(-0.9, 0.9)
    ax.set_yticks([])
    ax.tick_params(axis="x", labelsize=8)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

## Display — Monitor Window

In [ ]:
class InputMonitorWindow(QMainWindow):
    """Qt window showing four live AircraftCommand axis gauges.

    Keyboard events are captured via Qt keyPressEvent / keyReleaseEvent.
    Joystick is polled via pygame on a 20 Hz QTimer.  On joystick disconnect,
    reverts to keyboard (neutral until a key is pressed).
    """

    UPDATE_MS = 50  # 20 Hz

    _GAUGES = [
        # (field_name, v_min, v_max, v_neutral, label, unit)
        ("nz_g",            -2.0,  4.0,  1.0,  "Normal Load Factor   n_z",  "g"),
        ("ny_g",            -2.0,  2.0,  0.0,  "Lateral Load Factor  n_y",  "g"),
        ("roll_rate_deg_s", -90.0, 90.0, 0.0,  "Roll Rate",                 "°/s"),
        ("throttle_pct",    0.0,  100.0, 5.0,  "Throttle",                  "%"),
    ]

    def __init__(
        self,
        kbd_cfg: KeyboardConfig,
        joy: JoystickReader | None = None,
    ):
        super().__init__()
        self.kbd = KeyboardState(kbd_cfg)
        self.joy = joy

        # ---- Matplotlib figure embedded in Qt ----
        self.fig = mpl_figure.Figure(figsize=(9, 5), facecolor="#f5f5f5")
        self.fig.subplots_adjust(left=0.02, right=0.88, top=0.95, bottom=0.06, hspace=0.55)
        self.axes = [self.fig.add_subplot(4, 1, i + 1) for i in range(4)]
        self.canvas = FigureCanvasQTAgg(self.fig)

        # ---- Layout ----
        central = QWidget()
        layout = QVBoxLayout(central)
        layout.setContentsMargins(4, 4, 4, 4)
        layout.addWidget(self.canvas)

        help_label = QLabel(
            "Keyboard — ↑↓: n_z  ←→: roll  Q/E: n_y  W/S: throttle  Space: center"
        )
        help_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        help_label.setStyleSheet("color: #606060; font-size: 11px; padding: 2px;")
        layout.addWidget(help_label)

        self.setCentralWidget(central)
        self.setWindowTitle("Manual Input Verification — LiteAero Sim")
        self.resize(860, 540)

        self._source_label = QLabel("Input: KEYBOARD")
        self._source_label.setStyleSheet("padding-left: 6px; font-size: 12px;")
        self.statusBar().addWidget(self._source_label)

        # ---- Timer ----
        self._last_t = time.monotonic()
        self._timer = QTimer(self)
        self._timer.timeout.connect(self._tick)
        self._timer.start(self.UPDATE_MS)

    # -- Qt key events → KeyboardState --

    def keyPressEvent(self, event: QKeyEvent) -> None:
        if not event.isAutoRepeat():
            self.kbd.key_press(event.key())

    def keyReleaseEvent(self, event: QKeyEvent) -> None:
        if not event.isAutoRepeat():
            self.kbd.key_release(event.key())

    # -- Timer tick --

    def _tick(self) -> None:
        now = time.monotonic()
        dt_s = now - self._last_t
        self._last_t = now

        if self.joy is not None and self.joy.connected:
            self.joy.check_disconnect()

        if self.joy is not None and self.joy.connected:
            try:
                nz, ny, roll, throttle = self.joy.read()
                source = f"JOYSTICK  {self.joy.name}"
                src_color = "#1565C0"  # dark blue
            except Exception:
                self.joy.connected = False
                nz, ny, roll, throttle = 1.0, 0.0, 0.0, 0.05
                source = "KEYBOARD  (joystick disconnected — reverted to neutral)"
                src_color = "#2E7D32"  # dark green
        else:
            self.kbd.update(dt_s)
            nz       = self.kbd.nz_g
            ny       = self.kbd.ny_g
            roll     = self.kbd.roll_rate_rad_s
            throttle = self.kbd.throttle_nd
            source = "KEYBOARD"
            src_color = "#2E7D32"  # dark green

        self._source_label.setText(f"Input: {source}")
        self._draw(nz, ny, roll, throttle, src_color)

    # -- Drawing --

    def _draw(self, nz: float, ny: float, roll: float, throttle: float, color: str) -> None:
        values = {
            "nz_g":            nz,
            "ny_g":            ny,
            "roll_rate_deg_s": math.degrees(roll),
            "throttle_pct":    throttle * 100.0,
        }
        for ax, (field_name, v_min, v_max, v_neutral, label, unit) in zip(self.axes, self._GAUGES):
            draw_gauge(ax, values[field_name], v_min, v_max, v_neutral, label, unit, color)
        self.canvas.draw_idle()

---
## Run

Running this cell opens the monitor window.  Close the window to return to the notebook.

Joystick device 0 is opened automatically if present.  Edit `DEVICE_INDEX` and the
`JoystickConfig` axis indices above if your device uses a different layout.

In [ ]:
DEVICE_INDEX = 0  # change if you have multiple joysticks

# ---- Initialize pygame joystick subsystem (no display window) ----
pygame.joystick.init()

joy: JoystickReader | None = None
n_joy = pygame.joystick.get_count()
print(f"Joystick devices found: {n_joy}")
for i in range(n_joy):
    j = pygame.joystick.Joystick(i)
    j.init()
    print(f"  [{i}]  {j.get_name()}  ({j.get_numaxes()} axes, {j.get_numbuttons()} buttons)")
    j.quit()

if n_joy > 0:
    try:
        joy = JoystickReader(device_index=DEVICE_INDEX, cfg=JoystickConfig())
        print(f"\nOpened: {joy.name}  ({joy.axis_count} axes)")
        print("Raw axis values at rest (should be near 0):")
        print([f"{v:+.3f}" for v in joy.raw_axes()])
    except Exception as exc:
        print(f"Could not open joystick {DEVICE_INDEX}: {exc}")
else:
    print("No joystick — keyboard-only mode.")

# ---- Launch Qt window ----
app = QApplication.instance() or QApplication(sys.argv)
window = InputMonitorWindow(kbd_cfg=KeyboardConfig(), joy=joy)
window.show()
app.exec()

# ---- Cleanup ----
pygame.joystick.quit()